# Japan METI Demand Dashboard

METI petroleum statistics from `scripts/update_japan.py` →
`data/processed/japan/japan_meti_consumption.parquet`.

**Demand** (国内向販売, `TOTDEMO`): 年報 history + 確報 / 速報 updates.

**Supply balance** (確報 only): production, imports, exports, closing stocks (`CLOSTLV`) — Japan-specific vs other countries in this repo.

**Speed:** routine run pulls 速報 first, then 確報 replaces those months.

## Sections
1. Setup — load parquet; demand kL/t → kbd
2. **Headline total** (all products incl. **naphtha**)
3. Native products
4. Seasonality by year
5. METI vs JODI (by product)
6. **Total demand** — METI vs JODI
7. Jet fuel vs Kayrros
8. **Closing stocks** — levels and MoM change since Feb 2026 (US–Iran baseline)
9. **METI vs JODI (stocks)** — closing inventories (`CLOSTLV`, kb = KBBL)

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


def _resolve_project_root() -> Path:
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "scripts" / "update_japan.py").exists():
            return candidate
        if (candidate / "country_oil_scraper" / "scripts" / "update_japan.py").exists():
            return candidate / "country_oil_scraper"
    raise RuntimeError(f"Could not locate project root from cwd: {here}")


PROJECT_ROOT = _resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analytics.units import convert_series
from reference.japan import (
    DELIVERY_HEADLINE_NATIVE,
    DISPLAY_LABELS,
    INVENTORY_FOCUS_START,
    UNITS_KIND,
    build_demand_canonical,
    build_meti_jodi_panel_frames,
    build_meti_jodi_clostlv_panel_frames,
    jodi_compare_energy_products,
    jodi_compare_panels_present,
    JODI_COMPARE_PANEL_ORDER,
    seasonality_chart_inputs,
    seasonality_highlight_year,
)
from analytics import seasonality_by_year_chart

PARQUET = PROJECT_ROOT / "data" / "processed" / "japan" / "japan_meti_consumption.parquet"
df = pd.read_parquet(PARQUET)
df["date"] = pd.to_datetime(df["date"])
demand = df[df["metric_type"] == "TOTDEMO"].copy()
inventories = df[df["metric_type"] == "CLOSTLV"].copy()


def _to_kbd(frame: pd.DataFrame) -> pd.Series:
    """kL → kbd; tonnes (t) → kt → kbd using per-product density (UNITS_KIND)."""
    out = pd.Series(index=frame.index, dtype=float)
    sub = frame.assign(product_kind=frame["product_native"].map(UNITS_KIND))
    for (unit, kind), grp in sub.groupby(["unit", "product_kind"], dropna=False):
        m = grp.index
        if unit == "t":
            out.loc[m] = convert_series(
                grp["value"] / 1000,
                "kt",
                "kbd",
                product_kind=kind,
                date=grp["date"],
            )
        else:
            out.loc[m] = convert_series(
                grp["value"],
                "kL",
                "kbd",
                product_kind=kind,
                date=grp["date"],
            )
    return out


def _to_kb_stock(frame: pd.DataFrame) -> pd.Series:
    """Stock level (not rate): native kL/t → thousand barrels (JODI KBBL scale)."""
    out = pd.Series(index=frame.index, dtype=float)
    sub = frame.assign(product_kind=frame["product_native"].map(UNITS_KIND))
    for (unit, kind), grp in sub.groupby(["unit", "product_kind"], dropna=False):
        m = grp.index
        if unit == "t":
            out.loc[m] = convert_series(
                grp["value"] / 1000,
                "kt",
                "kb",
                product_kind=kind,
                date=grp["date"],
            )
        else:
            out.loc[m] = convert_series(
                grp["value"],
                "kL",
                "kb",
                product_kind=kind,
                date=grp["date"],
            )
    return out


_stock_products = list(DELIVERY_HEADLINE_NATIVE)
inventories_kb = inventories[
    inventories["product_native"].isin(_stock_products)
].copy()
if not inventories_kb.empty:
    inventories_kb["value_kb"] = _to_kb_stock(inventories_kb)

demand["product_kind"] = demand["product_native"].map(UNITS_KIND)
demand["value_kbd"] = _to_kbd(demand)
headline = demand[demand["product_native"].isin(DELIVERY_HEADLINE_NATIVE)]
demand_canonical = build_demand_canonical(demand, value_col="value_kbd")
print(f"Loaded {len(df):,} rows  {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  TOTDEMO: {len(demand):,}  |  CLOSTLV: {len(inventories):,}", end="")
if len(inventories):
    print(f"  ({inventories['date'].min().date()} → {inventories['date'].max().date()})")
else:
    print("  — run update_japan.py --reparse-all after 確報 xlsx are cached")
print(f"Provisional demand rows: {int(demand['is_provisional'].sum())}")
print(
    f"Canonical panels: {sorted(demand_canonical['product_canonical'].unique())}"
)

Loaded 4,153 rows  2007-01-01 → 2026-04-01
  TOTDEMO: 2,784  |  CLOSTLV: 360  (2023-11-01 → 2026-04-01)
Provisional demand rows: 12
Canonical panels: ['Bitumen', 'Diesel', 'Fuel oil', 'Gasoil', 'Gasoline', 'Grease', 'Jet Fuel', 'Kerosene', 'LPG', 'Lubricants', 'Naphtha', 'Wax']


In [2]:
df.to_csv('meti_data_complete.csv')

## 2. Headline total (incl. naphtha)

In [3]:
total = (
    headline.groupby(["date", "is_provisional"], as_index=False)["value_kbd"]
    .sum()
    .sort_values("date")
)
fig = px.line(
    total,
    x="date",
    y="value_kbd",
    color="is_provisional",
    title="Japan total domestic sales — METI (kbd, all headline products incl. naphtha)",
)
fig.show()

## 4. Seasonality

In [4]:
season_df, product_col, products, labels, suffix = seasonality_chart_inputs(
    "native",
    demand=demand,
    demand_canonical=demand_canonical,
)
if season_df.empty:
    print("No seasonality data")
else:
    fig = seasonality_by_year_chart(
        season_df,
        products=products,
        product_col=product_col,
        value_col="value_kbd",
        product_labels=labels,
        highlight_year=int(season_df['date'].dt.year.max()),#seasonality_highlight_year(season_df, date_col="date"),
        default_visible_prior_years=5,
        title=f"Seasonality by calendar year — Japan METI ({suffix}, kbd)",
        units_label="kbd",
    )
    fig.show()

## 5. METI vs JODI (by product)

**Gas/diesel oil** = METI 軽油 (`gas_oil`) + Ａ重油 (`fuel_oil_a`) summed vs JODI **GASDIES**. **Fuel oil** = Ｂ・Ｃ重油 only vs **RESFUEL**.

In [5]:
from analytics import cross_source_comparison_chart

JODI_PARQUET = PROJECT_ROOT / "data" / "processed" / "jodi" / "jodi_secondary.parquet"
if not JODI_PARQUET.exists():
    print(f"[skip] JODI parquet not found at {JODI_PARQUET}")
    print("       Run: python scripts/update_jodi.py")
else:
    jodi = pd.read_parquet(JODI_PARQUET)
    jodi["date"] = pd.to_datetime(jodi["date"])

    meti_panel, jodi_panel = build_meti_jodi_panel_frames(demand, jodi, value_col="value_kbd")
    panels = jodi_compare_panels_present(meti_panel)

    fig = cross_source_comparison_chart(
        df_a=meti_panel,
        df_b=jodi_panel,
        products=panels,
        product_col_a="panel",
        product_col_b="panel",
        value_col_a="value_kbd",
        value_col_b="value_kbd",
        label_a="METI (Japan)",
        label_b="JODI",
        title="Japan TOTDEMO — METI vs JODI (kbd)",
        units_label="kbd",
        cols=2,
        panel_height=280,
    )
    fig.show()

    cutoff_24 = meti_panel["date"].max() - pd.DateOffset(months=23)
    print("\nMean |gap| over last 24 months (kbd):")
    for panel in panels:
        m = meti_panel.loc[meti_panel["panel"] == panel].set_index("date")["value_kbd"]
        j = jodi_panel.loc[jodi_panel["panel"] == panel].set_index("date")["value_kbd"]
        merged = pd.concat([m, j], axis=1, keys=["meti", "jodi"]).dropna()
        merged = merged.loc[merged.index >= cutoff_24]
        if merged.empty:
            continue
        gap = (merged["meti"] - merged["jodi"]).abs().mean()
        pct = gap / merged["jodi"].abs().mean() * 100
        print(f"  {panel:12s}  mean|gap| = {gap:>8,.1f} kbd  ({pct:5.1f}% of JODI level)")


Mean |gap| over last 24 months (kbd):
  Gas/diesel oil  mean|gap| =      3.0 kbd  (  0.4% of JODI level)
  Fuel oil      mean|gap| =     88.5 kbd  ( 53.5% of JODI level)
  Gasoline      mean|gap| =      1.2 kbd  (  0.2% of JODI level)
  Kerosene      mean|gap| =     10.3 kbd  (  2.8% of JODI level)
  Jet fuel      mean|gap| =    167.7 kbd  ( 69.7% of JODI level)
  LPG           mean|gap| =      5.1 kbd  (  1.4% of JODI level)
  Naphtha       mean|gap| =      1.8 kbd  (  0.3% of JODI level)


## 6. Total demand — METI vs JODI

Sum of the **same seven product buckets** as section 5 (gasoline, naphtha, diesel/gas oil, jet, kerosene, LPG, fuel oil). A second chart compares METI **headline** domestic sales (incl. lubes, asphalt, etc.) to JODI **TOTPRODS**.

In [6]:
import plotly.graph_objects as go

JODI_PARQUET = PROJECT_ROOT / "data" / "processed" / "jodi" / "jodi_secondary.parquet"
if not JODI_PARQUET.exists():
    print(f"[skip] JODI parquet not found at {JODI_PARQUET}")
    print("       Run: python scripts/update_jodi.py")
else:
    jodi = pd.read_parquet(JODI_PARQUET)
    jodi["date"] = pd.to_datetime(jodi["date"])
    jodi_jp = jodi[
        (jodi["ref_area"] == "JP")
        & (jodi["flow_breakdown"] == "TOTDEMO")
        & (jodi["unit_measure"] == "KBD")
    ].copy()

    # METI / JODI totals — same scope as §5 (Gasoil + Diesel both → GASDIES once on JODI side)
    meti_panel, _jodi_panel = build_meti_jodi_panel_frames(demand, jodi, value_col="value_kbd")
    meti_compare_total = (
        meti_panel.groupby("date", as_index=False)["value_kbd"]
        .sum()
        .rename(columns={"value_kbd": "meti_kbd"})
    )
    jodi_compare_total = (
        jodi_jp[jodi_jp["energy_product"].isin(jodi_compare_energy_products())]
        .groupby("date", as_index=False)["obs_value"]
        .sum()
        .rename(columns={"obs_value": "jodi_kbd"})
    )

    cmp = meti_compare_total.merge(jodi_compare_total, on="date", how="inner")
    fig_cmp = go.Figure()
    fig_cmp.add_trace(
        go.Scatter(
            x=cmp["date"],
            y=cmp["meti_kbd"],
            name="METI (§5 product set)",
            line=dict(color="#1f77b4"),
        )
    )
    fig_cmp.add_trace(
        go.Scatter(
            x=cmp["date"],
            y=cmp["jodi_kbd"],
            name="JODI (§5 product set)",
            line=dict(color="#ff7f0e"),
        )
    )
    fig_cmp.update_layout(
        title="Total demand — METI vs JODI (§5 compare set; GASDIES counted once, kbd)",
        yaxis_title="kbd",
        template="plotly_white",
        hovermode="x unified",
        height=420,
    )
    fig_cmp.show()

    cutoff = cmp["date"].max() - pd.DateOffset(months=23)
    m = cmp[cmp["date"] >= cutoff]
    if not m.empty:
        gap = (m["meti_kbd"] - m["jodi_kbd"]).abs().mean()
        print(
            f"Comparable total — mean |gap| (24m): {gap:,.1f} kbd "
            f"({gap / m['jodi_kbd'].mean() * 100:.1f}% of JODI)"
        )

    # Broad: METI headline (all delivery products) vs JODI TOTPRODS
    meti_headline_total = (
        headline.groupby("date", as_index=False)["value_kbd"]
        .sum()
        .rename(columns={"value_kbd": "meti_kbd"})
    )
    jodi_totprods = (
        jodi_jp[jodi_jp["energy_product"] == "TOTPRODS"][["date", "obs_value"]]
        .rename(columns={"obs_value": "jodi_kbd"})
    )
    broad = meti_headline_total.merge(jodi_totprods, on="date", how="inner")
    fig_broad = go.Figure()
    fig_broad.add_trace(
        go.Scatter(
            x=broad["date"],
            y=broad["meti_kbd"],
            name="METI (headline incl. naphtha, lubes, asphalt)",
            line=dict(color="#1f77b4"),
        )
    )
    fig_broad.add_trace(
        go.Scatter(
            x=broad["date"],
            y=broad["jodi_kbd"],
            name="JODI TOTPRODS",
            line=dict(color="#ff7f0e"),
        )
    )
    fig_broad.update_layout(
        title="Broad total — METI headline vs JODI TOTPRODS (kbd)",
        yaxis_title="kbd",
        template="plotly_white",
        hovermode="x unified",
        height=420,
    )
    fig_broad.show()

    m2 = broad[broad["date"] >= cutoff]
    if not m2.empty:
        gap2 = (m2["meti_kbd"] - m2["jodi_kbd"]).abs().mean()
        print(
            f"Broad total — mean |gap| (24m): {gap2:,.1f} kbd "
            f"({gap2 / m2['jodi_kbd'].mean() * 100:.1f}% of JODI)"
        )
    print(
        "Note: §5 Gas/diesel oil = METI 軽油 + Ａ重油 vs JODI GASDIES; "
        "Fuel oil = Ｂ・Ｃ重油 vs RESFUEL. Broad METI adds lubes/asphalt/grease/wax."
    )

Comparable total — mean |gap| (24m): 153.7 kbd (5.4% of JODI)


Broad total — mean |gap| (24m): 471.1 kbd (14.4% of JODI)
Note: §5 Gas/diesel oil = METI 軽油 + Ａ重油 vs JODI GASDIES; Fuel oil = Ｂ・Ｃ重油 vs RESFUEL. Broad METI adds lubes/asphalt/grease/wax.


## 7. Jet fuel vs Kayrros

In [7]:
import os

KAYROS_ROOT = PROJECT_ROOT.parent / "kayros" / "jet_fuel"
DB_PATH = KAYROS_ROOT / "data" / "jet_fuel.duckdb"
if not DB_PATH.exists():
    print(f"[skip] Kayrros DB not found at {DB_PATH}")
    print("       Build/update kayros/jet_fuel/data/jet_fuel.duckdb first.")
else:
    if str(KAYROS_ROOT) not in sys.path:
        sys.path.insert(0, str(KAYROS_ROOT))
    os.environ.setdefault("JET_FUEL_DB_PATH", str(DB_PATH))
    from src.export import get_consumption

    jet = demand[demand["product_native"] == "jet_fuel"][["date", "value_kbd"]].sort_values("date")
    now_raw = get_consumption(
        scope_type="country",
        scope="Japan",
        freq="monthly",
        metric="avg_kbd",
        drop_incomplete=True,
    )
    now = now_raw.rename(columns={"period_start": "date", "value": "value_kbd"})
    plot = pd.concat(
        [
            jet.assign(label="METI — jet_fuel"),
            now[["date", "value_kbd"]].assign(label="Kayrros — flights"),
        ],
        ignore_index=True,
    )
    fig = px.line(
        plot,
        x="date",
        y="value_kbd",
        color="label",
        title="Japan jet fuel — METI vs Kayrros (kbd)",
    )
    fig.update_layout(height=440, template="plotly_white", hovermode="x unified")
    fig.show()

## 8. Closing stocks (CLOSTLV)

METI **在庫** from 確報 — month-end inventories in native **kL** (tonnes for LPG / asphalt / grease / paraffin), shown here as **mbbl** (thousand barrels → million bbl).

Baseline for the post–late-Feb 2026 window: **Feb 2026** closing level. MoM change = stock draw (negative) or build (positive).

The summary table has a **total** column (headline inventory change, same value on every row) and a **Total** row with Feb / latest levels.

History depth follows cached 確報 files (not 年報).

In [8]:
if inventories_kb.empty:
    print(
        "[skip] No CLOSTLV rows — run: python scripts/update_japan.py --no-download --reparse-all"
    )
else:
    WAR_BASELINE = INVENTORY_FOCUS_START
    stk = inventories_kb.copy()
    stk["value_mbbl"] = stk["value_kb"] / 1000.0
    stk = stk.sort_values(["product_native", "date"])
    stk["delta_mbbl"] = stk.groupby("product_native")["value_mbbl"].diff()

    baseline = (
        stk[stk["date"] == WAR_BASELINE]
        .set_index("product_native")["value_mbbl"]
    )
    latest_date = stk["date"].max()
    latest = stk[stk["date"] == latest_date].set_index("product_native")["value_mbbl"]

    chg = (latest - baseline).rename("change_mbbl_since_feb2026")
    summary = pd.DataFrame({"feb_2026_mbbl": baseline, "latest_mbbl": latest}).join(chg)
    summary["latest_month"] = latest_date.strftime("%Y-%m")
    summary.index = summary.index.map(lambda s: DISPLAY_LABELS.get(s, s))

    # Total row: sum stocks across headline products (matches the level chart below).
    by_date = stk.groupby("date", as_index=False)["value_mbbl"].sum()
    tot_feb = float(by_date.loc[by_date["date"] == WAR_BASELINE, "value_mbbl"].iloc[0])
    tot_latest = float(by_date.loc[by_date["date"] == latest_date, "value_mbbl"].iloc[0])
    total_row = pd.DataFrame(
        [
            {
                "feb_2026_mbbl": tot_feb,
                "latest_mbbl": tot_latest,
                "change_mbbl_since_feb2026": tot_latest - tot_feb,
                "latest_month": latest_date.strftime("%Y-%m"),
            }
        ],
        index=["Total"],
    )
    grand_change = tot_latest - tot_feb
    summary_sorted = summary.sort_values("change_mbbl_since_feb2026")
    summary_sorted["total"] = grand_change
    total_row["total"] = grand_change
    summary_out = pd.concat([summary_sorted, total_row])
    display(summary_out.round(3))
    print(
        f"Total inventory change (Feb 2026 → {latest_date:%Y-%m}): "
        f"{total_row['change_mbbl_since_feb2026'].iloc[0]:+.3f} mbbl"
    )

    total = stk.groupby("date", as_index=False)["value_mbbl"].sum()
    total = total.sort_values("date")
    total["delta_mbbl"] = total["value_mbbl"].diff()

    recent = total[total["date"] >= "2025-10-01"]
    fig = px.line(
        recent,
        x="date",
        y="value_mbbl",
        title="Japan total product stocks (mbbl) — METI 在庫",
        markers=True,
    )
    fig.add_shape(
        type="line",
        x0=WAR_BASELINE,
        x1=WAR_BASELINE,
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color="gray", width=1, dash="dash"),
    )
    fig.add_annotation(
        x=WAR_BASELINE,
        y=1.02,
        xref="x",
        yref="paper",
        text="Feb 2026 baseline",
        showarrow=False,
        font=dict(color="gray", size=11),
    )
    fig.update_layout(height=420, template="plotly_white", yaxis_title="mbbl")
    fig.show()

    mom = stk[stk["date"] >= "2025-10-01"].copy()
    mom["product"] = mom["product_native"].map(DISPLAY_LABELS)
    fig2 = px.bar(
        mom,
        x="date",
        y="delta_mbbl",
        color="product",
        barmode="relative",
        title="MoM stock change by product (mbbl)",
    )
    fig2.update_layout(height=460, template="plotly_white")
    fig2.show()

,feb_2026_mbbl,latest_mbbl,change_mbbl_since_feb2026,latest_month,total
LPG,19.178,14.974,-4.203,2026-04,-6.717
Kerosene,9.383,7.546,-1.837,2026-04,-6.717
Naphtha,8.797,7.774,-1.023,2026-04,-6.717
Gas oil (diesel),8.641,8.319,-0.322,2026-04,-6.717
Lubricating oil,2.507,2.369,-0.138,2026-04,-6.717
Asphalt,1.187,1.140,-0.047,2026-04,-6.717
Fuel oil A (gasoil),4.430,4.386,-0.043,2026-04,-6.717
Fuel oil B·C (heavy),7.601,7.559,-0.041,2026-04,-6.717
Grease,0.085,0.077,-0.008,2026-04,-6.717
Paraffin wax,0.074,0.066,-0.008,2026-04,-6.717


Total inventory change (Feb 2026 → 2026-04): -6.717 mbbl


## 9. METI vs JODI (CLOSTLV, kb)

Side-by-side **closing stocks** from METI 在庫 (確報 + 速報) vs JODI secondary `CLOSTLV` for Japan (`JP`). Both are in **thousand barrels** (METI kL/t → `value_kb`; JODI `KBBL`).

Same product panels as §5: gas/diesel oil = 軽油 + Ａ重油 vs **GASDIES**; fuel oil = Ｂ・Ｃ重油 vs **RESFUEL**.

METI inventory history starts when 確報 files are cached (~2023-11); JODI runs from 2002. Latest METI month can be one ahead of JODI (e.g. Apr 2026 速報).

In [9]:
from analytics import cross_source_comparison_chart

JODI_PARQUET = PROJECT_ROOT / "data" / "processed" / "jodi" / "jodi_secondary.parquet"
if inventories_kb.empty:
    print("[skip] No CLOSTLV rows — run: python scripts/update_japan.py --reparse-all")
elif not JODI_PARQUET.exists():
    print(f"[skip] JODI parquet not found at {JODI_PARQUET}")
    print("       Run: python scripts/update_jodi.py")
else:
    jodi = pd.read_parquet(JODI_PARQUET)
    jodi["date"] = pd.to_datetime(jodi["date"])

    meti_stk, jodi_stk = build_meti_jodi_clostlv_panel_frames(
        inventories_kb, jodi, value_col="value_kb"
    )
    panels = jodi_compare_panels_present(meti_stk)

    fig = cross_source_comparison_chart(
        df_a=meti_stk,
        df_b=jodi_stk,
        products=panels,
        product_col_a="panel",
        product_col_b="panel",
        value_col_a="value_kb",
        value_col_b="value_kb",
        label_a="METI (Japan)",
        label_b="JODI",
        title="Japan CLOSTLV — METI vs JODI (kb = KBBL)",
        units_label="kb",
        cols=2,
        panel_height=280,
    )
    fig.show()

    overlap_start = meti_stk["date"].min()
    cutoff_24 = meti_stk["date"].max() - pd.DateOffset(months=23)
    print(f"\nMean |gap| over overlap ({overlap_start:%Y-%m} → latest, kb):")
    for panel in panels:
        m = meti_stk.loc[meti_stk["panel"] == panel].set_index("date")["value_kb"]
        j = jodi_stk.loc[jodi_stk["panel"] == panel].set_index("date")["value_kb"]
        merged = pd.concat([m, j], axis=1, keys=["meti", "jodi"]).dropna()
        merged = merged.loc[merged.index >= max(cutoff_24, overlap_start)]
        if merged.empty:
            continue
        gap = (merged["meti"] - merged["jodi"]).abs().mean()
        pct = gap / merged["jodi"].abs().mean() * 100
        print(f"  {panel:14s}  mean|gap| = {gap:>10,.0f} kb  ({pct:5.1f}% of JODI level)")

    # Latest overlapping month — level differences by product
    last_overlap = min(meti_stk["date"].max(), jodi_stk["date"].max())
    m_last = (
        meti_stk[meti_stk["date"] == last_overlap]
        .set_index("panel")["value_kb"]
        .rename("meti_kb")
    )
    j_last = (
        jodi_stk[jodi_stk["date"] == last_overlap]
        .set_index("panel")["value_kb"]
        .rename("jodi_kb")
    )
    diff = pd.DataFrame({"meti_kb": m_last, "jodi_kb": j_last}).join(
        how="outer"
    )
    diff["gap_kb"] = diff["meti_kb"] - diff["jodi_kb"]
    diff["gap_pct"] = 100 * diff["gap_kb"] / diff["jodi_kb"]
    diff = diff.loc[[p for p in JODI_COMPARE_PANEL_ORDER if p in diff.index]]
    print(f"\nLevels at {last_overlap:%Y-%m} (kb):")
    display(diff.round(1))

    # Headline totals (§9 product set)
    meti_tot = meti_stk.groupby("date", as_index=False)["value_kb"].sum()
    jodi_tot = jodi_stk.groupby("date", as_index=False)["value_kb"].sum()
    cmp = meti_tot.merge(jodi_tot, on="date", how="inner", suffixes=("_meti", "_jodi"))

    fig2 = go.Figure()
    fig2.add_trace(
        go.Scatter(
            x=cmp["date"],
            y=cmp["value_kb_meti"],
            name="METI (§9 product set)",
            line=dict(color="#1f77b4"),
        )
    )
    fig2.add_trace(
        go.Scatter(
            x=cmp["date"],
            y=cmp["value_kb_jodi"],
            name="JODI (§9 product set)",
            line=dict(color="#ff7f0e"),
        )
    )
    fig2.update_layout(
        title="Total closing stocks — METI vs JODI (same products as panels, kb)",
        height=420,
        template="plotly_white",
        yaxis_title="kb",
        hovermode="x unified",
    )
    fig2.show()

    jodi_totprods = jodi[
        (jodi["ref_area"] == "JP")
        & (jodi["flow_breakdown"] == "CLOSTLV")
        & (jodi["unit_measure"] == "KBBL")
        & (jodi["energy_product"] == "TOTPRODS")
    ][["date", "obs_value"]].rename(columns={"obs_value": "jodi_kb"})

    row = cmp[cmp["date"] == last_overlap].iloc[0]
    tp = jodi_totprods[jodi_totprods["date"] == last_overlap]
    jodi_headline = float(tp["jodi_kb"].iloc[0]) if len(tp) else float("nan")
    print(f"\nAt {last_overlap:%Y-%m} (comparable product total, kb):")
    print(f"  METI (§9 set)  : {row['value_kb_meti']:,.0f}")
    print(f"  JODI (§9 set)  : {row['value_kb_jodi']:,.0f}")
    print(f"  METI − JODI    : {row['value_kb_meti'] - row['value_kb_jodi']:+,.0f}")
    if len(tp):
        print(f"  JODI TOTPRODS  : {jodi_headline:,.0f}")
        print(f"  METI − TOTPRODS: {row['value_kb_meti'] - jodi_headline:+,.0f}")

    base = INVENTORY_FOCUS_START
    if base in set(cmp["date"]) and last_overlap >= base:
        b = cmp[cmp["date"] == base].iloc[0]
        l = cmp[cmp["date"] == last_overlap].iloc[0]
        print(f"\nChange {base:%Y-%m} → {last_overlap:%Y-%m} (§9 product total, kb):")
        print(f"  METI : {l['value_kb_meti'] - b['value_kb_meti']:+,.0f}")
        print(f"  JODI : {l['value_kb_jodi'] - b['value_kb_jodi']:+,.0f}")


Mean |gap| over overlap (2023-11 → latest, kb):
  Gas/diesel oil  mean|gap| =      3,305 kb  ( 19.1% of JODI level)
  Fuel oil        mean|gap| =        359 kb  (  4.9% of JODI level)
  Gasoline        mean|gap| =      3,530 kb  ( 25.2% of JODI level)
  Kerosene        mean|gap| =      2,298 kb  ( 19.3% of JODI level)
  Jet fuel        mean|gap| =         62 kb  (  1.3% of JODI level)
  LPG             mean|gap| =     16,156 kb  ( 45.1% of JODI level)
  Naphtha         mean|gap| =         36 kb  (  0.4% of JODI level)


TypeError: DataFrame.join() missing 1 required positional argument: 'other'